In [1]:
import numpy as np
import pandas as pd
import os
import copy
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")

# ==================== AF MODEL DEFINITION ====================

class AFFeatureExtractor(nn.Module):
    """
    مدل AF با معماری MLP (مناسب برای داده جدولی)
    """
    def __init__(self, input_dim, num_classes, embedding_size=512, hidden_dims=[256, 128]):
        super().__init__()
        
        self.input_dim = input_dim
        self.embedding_size = embedding_size
        
        # ========== Feature Extractor (MLP با BatchNorm و Dropout) ==========
        feature_layers = []
        prev_dim = input_dim
        
        for i, h_dim in enumerate(hidden_dims):
            feature_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ELU(),
                nn.Dropout(0.1)
            ])
            prev_dim = h_dim
        
        # Embedding layer
        feature_layers.extend([
            nn.Linear(prev_dim, embedding_size),
            nn.BatchNorm1d(embedding_size)
        ])
        
        self.feature_extractor = nn.Sequential(*feature_layers)
        
        # ========== Classifier (روی بردار ویژگی) ==========
        self.classifier = nn.Sequential(
            nn.Linear(embedding_size, 512),
            nn.BatchNorm1d(512),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
            nn.BatchNorm1d(num_classes)
        )
        
        # ========== Discriminator (روی بردار ویژگی) ==========
        self.discriminator = nn.Sequential(
            nn.Linear(embedding_size, 512),
            nn.BatchNorm1d(512),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
        # جدا کردن پارامترها برای بهینه‌سازهای مختلف
        self.classifier_params = list(self.classifier.parameters())
        self.discriminator_params = list(self.discriminator.parameters())
        self.feature_params = list(self.feature_extractor.parameters())
        
        # نرخ‌های یادگیری متفاوت
        self.lr_classifier = 1e-4
        self.lr_discriminator = 1e-5
        self.lr_combined = 1e-5
        
        self.class_loss_weight = 4.0
        self.dis_loss_weight = 4.0
    
    def forward(self, x, return_features=False):
        """forward معمولی برای inference"""
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        class_output = nn.functional.softmax(class_output, dim=1)
        
        if return_features:
            return class_output, features
        return class_output
    
    def forward_with_domain(self, x):
        """forward با خروجی طبقه‌بند و تمایزگر (برای آموزش)"""
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        domain_output = self.discriminator(features)
        return class_output, domain_output
    
    def predict(self, x, device='cpu'):
        """پیش‌بینی برای یک یا چند نمونه"""
        self.eval()
        with torch.no_grad():
            if len(x.shape) == 1:
                x_tensor = torch.FloatTensor(x).unsqueeze(0).to(device)
            else:
                x_tensor = torch.FloatTensor(x).to(device)
            class_output = self.forward(x_tensor)
            pred = torch.argmax(class_output, dim=1).cpu().numpy()
            if len(pred) == 1:
                return pred[0]
            return pred
    
    def partial_fit_batch(self, X_source, y_source, X_target, device='cpu', 
                          class_loss_weight=4.0, dis_loss_weight=4.0):
        """آموزش یک بچ به روش ADA"""
        self.train()
        
        X_s = torch.FloatTensor(X_source).to(device)
        y_s = torch.LongTensor(y_source).to(device)
        X_t = torch.FloatTensor(X_target).to(device)
        
        batch_size = min(X_s.size(0), X_t.size(0))
        X_s = X_s[:batch_size]
        y_s = y_s[:batch_size]
        X_t = X_t[:batch_size]
        
        X_combined = torch.cat([X_s, X_t], dim=0)
        
        # مرحله 1: ذخیره وزن‌های discriminator
        disc_weights = copy.deepcopy(self.discriminator.state_dict())
        
        # مرحله 2: آموزش combined model
        opt_combined = optim.AdamW(self.feature_params + self.classifier_params, 
                                   lr=self.lr_combined, betas=(0.9, 0.999), weight_decay=1e-4)
        opt_combined.zero_grad()
        
        class_output, domain_output = self.forward_with_domain(X_combined)
        
        # Classifier loss (فقط روی داده منبع)
        class_loss = nn.functional.cross_entropy(class_output[:batch_size], y_s)
        
        # Domain loss برای combined
        domain_labels_combined = torch.cat([
            torch.ones(batch_size, dtype=torch.float32).to(device),
            torch.zeros(batch_size, dtype=torch.float32).to(device)
        ]).unsqueeze(1)
        domain_loss_combined = nn.functional.binary_cross_entropy(domain_output, domain_labels_combined)
        
        total_loss_combined = class_loss_weight * class_loss - dis_loss_weight * domain_loss_combined
        total_loss_combined.backward()
        torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=0.5)
        opt_combined.step()
        
        # مرحله 3: برگرداندن وزن‌های discriminator
        self.discriminator.load_state_dict(disc_weights)
        
        # مرحله 4: آموزش جداگانه discriminator
        opt_discriminator = optim.AdamW(self.discriminator_params, lr=self.lr_discriminator,
                                        betas=(0.9, 0.999), weight_decay=1e-4)
        opt_discriminator.zero_grad()
        
        with torch.no_grad():
            features_s = self.feature_extractor(X_s)
            features_t = self.feature_extractor(X_t)
        
        features_combined = torch.cat([features_s, features_t], dim=0)
        domain_output_new = self.discriminator(features_combined)
        
        domain_labels_disc = torch.cat([
            torch.zeros(batch_size, dtype=torch.float32).to(device),
            torch.ones(batch_size, dtype=torch.float32).to(device)
        ]).unsqueeze(1)
        domain_loss_disc = nn.functional.binary_cross_entropy(domain_output_new, domain_labels_disc)
        domain_loss_disc.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator_params, max_norm=0.5)
        opt_discriminator.step()
        
        return {
            'class_loss': class_loss.item(),
            'domain_loss_combined': domain_loss_combined.item(),
            'domain_loss_disc': domain_loss_disc.item()
        }


class AFClassifier:
    """Wrapper کلاس AF با قابلیت ذخیره و بارگذاری"""
    
    def __init__(self, input_dim=None, num_classes=None, embedding_size=512, 
                 hidden_dims=[256, 128], seed=42, 
                 class_loss_weight=4.0, dis_loss_weight=4.0):
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.embedding_size = embedding_size
        self.hidden_dims = hidden_dims
        self.seed = seed
        self.class_loss_weight = class_loss_weight
        self.dis_loss_weight = dis_loss_weight
        self.model = None
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.is_fitted = False
        self.scaler = None
        self.label_encoder = None
        
        # بافر برای آموزش آنلاین
        self.buffer_X = []
        self.buffer_y = []
        self.buffer_X_target = []
    
    def _initialize_model(self):
        """ایجاد مدل"""
        if self.input_dim is None or self.num_classes is None:
            raise ValueError("input_dim and num_classes must be set")
        
        torch.manual_seed(self.seed)
        self.model = AFFeatureExtractor(
            input_dim=self.input_dim,
            num_classes=self.num_classes,
            embedding_size=self.embedding_size,
            hidden_dims=self.hidden_dims
        ).to(self.device)
        self.is_fitted = True
        
        print(f"  Model initialized: input_dim={self.input_dim}, num_classes={self.num_classes}")
    
    def set_preprocessors(self, scaler, label_encoder):
        """ذخیره scaler و label encoder"""
        self.scaler = scaler
        self.label_encoder = label_encoder
    
    def learn_one(self, x, y):
        """یادگیری از یک نمونه (online learning با بافر)"""
        if not self.is_fitted:
            self.input_dim = len(x)
            self.num_classes = max(y + 1, 2)
            self._initialize_model()
        
        if isinstance(x, dict):
            x = np.array([x[i] for i in range(len(x))])
        
        self.buffer_X.append(x)
        self.buffer_y.append(y)
        self.buffer_X_target.append(x)
        
        if len(self.buffer_X) >= 32:
            X_batch = np.array(self.buffer_X)
            y_batch = np.array(self.buffer_y)
            X_target_batch = np.array(self.buffer_X_target)
            
            _ = self.model.partial_fit_batch(
                X_batch, y_batch, X_target_batch, 
                device=self.device,
                class_loss_weight=self.class_loss_weight,
                dis_loss_weight=self.dis_loss_weight
            )
            
            self.buffer_X = []
            self.buffer_y = []
            self.buffer_X_target = []
        
        return self
    
    def predict_one(self, x):
        """پیش‌بینی برای یک نمونه"""
        if not self.is_fitted:
            return 0
        
        if isinstance(x, dict):
            x = np.array([x[i] for i in range(len(x))])
        
        return self.model.predict(x, device=self.device)
    
    def predict_batch(self, X):
        """پیش‌بینی برای دسته‌ای از نمونه‌ها"""
        if not self.is_fitted:
            return np.zeros(len(X))
        
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.model.forward(X_tensor)
            y_pred = torch.argmax(predictions, dim=1).cpu().numpy()
        return y_pred
    
    def train_batch_mode(self, X_source, y_source, X_target, n_epochs=80, batch_size=32):
        """آموزش بچ (برای مدل پایه)"""
        if not self.is_fitted:
            self.input_dim = X_source.shape[1]
            self.num_classes = len(np.unique(y_source))
            self._initialize_model()
        
        n_samples = len(X_source)
        
        print(f"  Training: {n_samples} samples, batch_size={batch_size}, epochs={n_epochs}")
        
        for epoch in range(n_epochs):
            indices = np.random.permutation(n_samples)
            total_class_loss = 0
            total_domain_loss_combined = 0
            total_domain_loss_disc = 0
            n_batches = 0
            
            for i in range(0, n_samples, batch_size):
                batch_indices = indices[i:i+batch_size]
                X_s_batch = X_source[batch_indices]
                y_s_batch = y_source[batch_indices]
                
                target_indices = np.random.choice(n_samples, len(batch_indices), replace=False)
                X_t_batch = X_source[target_indices]
                
                losses = self.model.partial_fit_batch(
                    X_s_batch, y_s_batch, X_t_batch,
                    device=self.device,
                    class_loss_weight=self.class_loss_weight,
                    dis_loss_weight=self.dis_loss_weight
                )
                
                total_class_loss += losses['class_loss']
                total_domain_loss_combined += losses['domain_loss_combined']
                total_domain_loss_disc += losses['domain_loss_disc']
                n_batches += 1
            
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1:3d}: Class Loss={total_class_loss/n_batches:.4f}, "
                      f"Domain Comb={total_domain_loss_combined/n_batches:.4f}, "
                      f"Domain Disc={total_domain_loss_disc/n_batches:.4f}")
        
        return self
    
    def save_model(self, model_path='saved_models'):
        """ذخیره مدل و پیش‌پردازشگرها"""
        if not os.path.exists(model_path):
            os.makedirs(model_path)
        
        # ذخیره state_dict مدل
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'input_dim': self.input_dim,
            'num_classes': self.num_classes,
            'embedding_size': self.embedding_size,
            'hidden_dims': self.hidden_dims,
            'seed': self.seed,
            'class_loss_weight': self.class_loss_weight,
            'dis_loss_weight': self.dis_loss_weight,
        }, f'{model_path}/af_model.pth')
        
        # ذخیره scaler و label_encoder
        if self.scaler is not None:
            with open(f'{model_path}/af_scaler.pkl', 'wb') as f:
                pickle.dump(self.scaler, f)
        
        if self.label_encoder is not None:
            with open(f'{model_path}/af_label_encoder.pkl', 'wb') as f:
                pickle.dump(self.label_encoder, f)
        
        print(f"  Model saved to {model_path}/")
    
    def load_model(self, model_path='saved_models'):
        """بارگذاری مدل و پیش‌پردازشگرها"""
        # بارگذاری مدل
        checkpoint = torch.load(f'{model_path}/af_model.pth', map_location=self.device)
        
        self.input_dim = checkpoint['input_dim']
        self.num_classes = checkpoint['num_classes']
        self.embedding_size = checkpoint['embedding_size']
        self.hidden_dims = checkpoint['hidden_dims']
        self.seed = checkpoint['seed']
        self.class_loss_weight = checkpoint['class_loss_weight']
        self.dis_loss_weight = checkpoint['dis_loss_weight']
        
        self._initialize_model()
        self.model.load_state_dict(checkpoint['model_state_dict'])
        
        # بارگذاری scaler و label_encoder
        if os.path.exists(f'{model_path}/af_scaler.pkl'):
            with open(f'{model_path}/af_scaler.pkl', 'rb') as f:
                self.scaler = pickle.load(f)
        
        if os.path.exists(f'{model_path}/af_label_encoder.pkl'):
            with open(f'{model_path}/af_label_encoder.pkl', 'rb') as f:
                self.label_encoder = pickle.load(f)
        
        print(f"  Model loaded from {model_path}/")
        return self


# ==================== DATA LOADING FUNCTIONS ====================

def load_data(dataset_path='Dataset', csv_filename='UNSW_IoT_original.csv'):
    """بارگذاری داده"""
    original_df = pd.read_csv(f'{dataset_path}/{csv_filename}')
    train_df, temp_df = train_test_split(original_df, test_size=0.4, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    return train_df, val_df, test_df


def preprocess_data(train_df, val_df, test_df=None):
    """پیش‌پردازش داده‌ها"""
    X_train = train_df.iloc[:, :-1].values
    y_train = train_df.iloc[:, -1].values
    X_val = val_df.iloc[:, :-1].values
    y_val = val_df.iloc[:, -1].values
    
    # Encoding
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)
    
    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    result = {
        'X_train': X_train_scaled,
        'y_train': y_train_encoded,
        'X_val': X_val_scaled,
        'y_val': y_val_encoded,
        'scaler': scaler,
        'label_encoder': label_encoder
    }
    
    if test_df is not None:
        X_test = test_df.iloc[:, :-1].values
        y_test = test_df.iloc[:, -1].values
        result['X_test'] = scaler.transform(X_test)
        result['y_test'] = label_encoder.transform(y_test)
    
    return result


def load_perturbation_data(base_path='Dataset', attack_type='FGSM', test_perturb_levels=[0.1, 0.5, 1.0, 2.0, 5.0], 
                          scaler=None, label_encoder=None):
    """بارگذاری داده‌های perturbed"""
    datasets = []
    
    for perturb_level in sorted(test_perturb_levels):
        file_path = f'{base_path}/{attack_type}/{attack_type}_eps_{perturb_level}.csv'
        if not os.path.exists(file_path):
            print(f"Warning: {file_path} not found, skipping...")
            continue
            
        df = pd.read_csv(file_path)
        X = df.iloc[:, :-1].values
        y = df.iloc[:, -1].values
        
        y_encoded = label_encoder.transform(y)
        X_scaled = scaler.transform(X)
        
        datasets.append({
            'X': X_scaled,
            'y': y_encoded,
            'perturb_level': perturb_level
        })
        print(f"  Loaded {attack_type} eps={perturb_level}: {len(X_scaled)} samples")
    
    return datasets


# ==================== TRAINING BLOCK ====================

def train_model(attack_type='FGSM', save_model_path='saved_models'):
    """
    بلوک آموزش - فقط برای آموزش مدل پایه استفاده می‌شود
    """
    print("\n" + "="*80)
    print("TRAINING BLOCK - AF Model with ADA (Adversarial Domain Adaptation)")
    print("="*80)
    
    # بارگذاری داده
    print("\n[1] Loading training data...")
    train_df, val_df, _ = load_data()
    print(f"  Train: {len(train_df)}, Val: {len(val_df)}")
    
    # پیش‌پردازش
    print("\n[2] Preprocessing data...")
    preprocessed = preprocess_data(train_df, val_df)
    
    X_train = preprocessed['X_train']
    y_train = preprocessed['y_train']
    X_val = preprocessed['X_val']
    y_val = preprocessed['y_val']
    scaler = preprocessed['scaler']
    label_encoder = preprocessed['label_encoder']
    
    input_dim = X_train.shape[1]
    num_classes = len(label_encoder.classes_)
    
    print(f"\n[3] Initializing AF Model...")
    print(f"  Input dimension: {input_dim}")
    print(f"  Number of classes: {num_classes}")
    print(f"  Embedding size: 512")
    print(f"  Hidden dims: [256, 128]")
    print(f"  Class loss weight: {4.0}, Domain loss weight: {4.0}")
    print(f"  Learning rates: Classifier=1e-4, Discriminator=1e-5, Combined=1e-5")
    
    # ایجاد مدل
    model = AFClassifier(
        input_dim=input_dim,
        num_classes=num_classes,
        embedding_size=512,
        hidden_dims=[256, 128],
        class_loss_weight=4.0,
        dis_loss_weight=4.0,
        seed=42
    )
    
    # آموزش
    print("\n[4] Training model...")
    model.train_batch_mode(X_train, y_train, X_train, n_epochs=80, batch_size=32)
    
    # ارزیابی روی validation
    print("\n[5] Evaluating on validation set...")
    y_val_pred = model.predict_batch(X_val)
    
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_precision = precision_score(y_val, y_val_pred, average='weighted', zero_division=0)
    val_recall = recall_score(y_val, y_val_pred, average='weighted', zero_division=0)
    val_f1 = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)
    
    print(f"\n{'='*60}")
    print(f"VALIDATION RESULTS")
    print(f"{'='*60}")
    print(f"  Accuracy:  {val_accuracy:.4f}")
    print(f"  Precision: {val_precision:.4f}")
    print(f"  Recall:    {val_recall:.4f}")
    print(f"  F1-Score:  {val_f1:.4f}")
    
    # ذخیره پیش‌پردازشگرها در مدل
    model.set_preprocessors(scaler, label_encoder)
    
    # ذخیره مدل
    print(f"\n[6] Saving model...")
    model.save_model(save_model_path)
    
    print("\n" + "="*60)
    print("TRAINING COMPLETED SUCCESSFULLY")
    print("="*60)
    
    return model


# ==================== TESTING BLOCK ====================

def test_model(attack_type='FGSM', test_perturb_levels=[0.1, 0.5, 1.0, 2.0, 5.0], 
               update_budget=0.8, model_path='Models'):
    """
    بلوک تست - بارگذاری مدل ذخیره شده و تست روی داده‌های perturbed
    """
    print("\n" + "="*80)
    print("TESTING BLOCK - AF Model with ADA")
    print(f"Attack Type: {attack_type}, Update Budget: {update_budget*100}%")
    print("="*80)
    
    # بارگذاری مدل
    print("\n[1] Loading pre-trained model...")
    model = AFClassifier()
    model.load_model(model_path)
    
    if model.scaler is None or model.label_encoder is None:
        print("Error: Scaler or LabelEncoder not found in saved model!")
        return None
        
    # بارگذاری داده‌های perturbed
    print(f"\n[3] Loading perturbed data (attack type: {attack_type})...")
    perturb_datasets = load_perturbation_data(
        base_path='Dataset',
        attack_type=attack_type,
        test_perturb_levels=test_perturb_levels,
        scaler=model.scaler,
        label_encoder=model.label_encoder
    )
    
    if len(perturb_datasets) == 0:
        print(f"No perturbed datasets found for attack type: {attack_type}")
        return None
    
    # تست آنلاین
    print(f"\n[4] Running online tests with {update_budget*100}% update budget...")
    results = []
    
    for dataset in perturb_datasets:
        result = test_online_with_budget(
            model,
            dataset['X'],
            dataset['y'],
            dataset['perturb_level'],
            update_budget
        )
        results.append(result)
    
    # ذخیره نتایج
    df_results = pd.DataFrame(results)
    print("\n" + "="*60)
    print("FINAL RESULTS SUMMARY (4 Metrics)")
    print("="*60)
    print(df_results[['Perturbation Level', 'Updates Done', 'Accuracy', 'Precision', 'Recall', 'F1 Score']].to_string(index=False))
    
    if not os.path.exists('Results'):
        os.makedirs('Results')
    
    output_file = f'Results/af_ada_{attack_type}_budget_{int(update_budget*100)}.csv'
    df_results.to_csv(output_file, index=False)
    print(f"\nResults saved to: {output_file}")
    
    return df_results


def test_online_with_budget(model, X_test, y_test, perturb_level, update_budget=1.0):
    """تست آنلاین با بودجه"""
    test_model = copy.deepcopy(model)
    
    n_samples = len(X_test)
    n_updates_allowed = int(n_samples * update_budget)
    
    y_pred = []
    y_true_list = []
    updates_done = 0
    
    print(f"\n--- Testing Perturbation Level {perturb_level} (Budget: {update_budget*100}%) ---")
    
    for i, (xi, yi) in enumerate(zip(X_test, y_test)):
        xi_dict = {j: float(x) for j, x in enumerate(xi)}
        
        pred = test_model.predict_one(xi_dict)
        y_pred.append(pred)
        y_true_list.append(yi)
        
        if updates_done < n_updates_allowed:
            test_model.learn_one(xi_dict, yi)
            updates_done += 1
        
        if (i + 1) % 1000 == 0 and i > 0:
            current_acc = accuracy_score(y_true_list[-1000:], y_pred[-1000:])
            print(f"  Sample {i+1}: Updates={updates_done}, Recent Acc={current_acc:.4f}")
    
    # محاسبه 4 سنجه اصلی
    accuracy = accuracy_score(y_true_list, y_pred)
    precision = precision_score(y_true_list, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true_list, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true_list, y_pred, average='weighted', zero_division=0)
    
    print(f"  Final - Acc: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    
    return {
        'Perturbation Level': perturb_level,
        'Update Budget %': update_budget * 100,
        'Total Samples': n_samples,
        'Updates Done': updates_done,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    }


def test_multiple_budgets(attack_type='FGSM', perturb_levels=[0.1, 0.5, 1.0, 2.0, 5.0], model_path='saved_models'):
    """تست با چند بودجه مختلف"""
    budgets = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    all_results = []
    
    print("\n" + "="*80)
    print("TESTING BLOCK - MULTIPLE BUDGETS (4 Metrics)")
    print("="*80)
    
    # بارگذاری مدل (یک بار)
    print("\n[1] Loading pre-trained model...")
    base_model = AFClassifier()
    base_model.load_model(model_path)
    
    if base_model.scaler is None or base_model.label_encoder is None:
        print("Error: Scaler or LabelEncoder not found in saved model!")
        return None
    
    # بارگذاری داده‌های perturbed
    print(f"\n[2] Loading perturbed data (attack type: {attack_type})...")
    perturb_datasets = load_perturbation_data(
        attack_type=attack_type,
        test_perturb_levels=perturb_levels,
        scaler=base_model.scaler,
        label_encoder=base_model.label_encoder
    )
    
    if len(perturb_datasets) == 0:
        print(f"No perturbed datasets found for attack type: {attack_type}")
        return None
    
    for budget in budgets:
        print(f"\n{'='*60}")
        print(f"BUDGET: {budget*100}%")
        print('='*60)
        
        for dataset in perturb_datasets:
            result = test_online_with_budget(
                base_model,
                dataset['X'],
                dataset['y'],
                dataset['perturb_level'],
                budget
            )
            all_results.append(result)
            print(f"  Level {result['Perturbation Level']}: Acc={result['Accuracy']:.4f}, "
                  f"Prec={result['Precision']:.4f}, Rec={result['Recall']:.4f}, F1={result['F1 Score']:.4f}")
    
    # ذخیره و خلاصه
    df_all = pd.DataFrame(all_results)
    os.makedirs('Results', exist_ok=True)
    df_all.to_csv(f'Results/af_ada_{attack_type}_all_budgets.csv', index=False)
    
    # خلاصه Accuracy
    summary_acc = df_all.pivot_table(
        index='Perturbation Level',
        columns='Update Budget %',
        values='Accuracy',
        aggfunc='first'
    )
    
    # خلاصه F1
    summary_f1 = df_all.pivot_table(
        index='Perturbation Level',
        columns='Update Budget %',
        values='F1 Score',
        aggfunc='first'
    )
    
    print("\n" + "="*60)
    print("ACCURACY SUMMARY TABLE")
    print("="*60)
    print(summary_acc.round(4))
    
    print("\n" + "="*60)
    print("F1 SCORE SUMMARY TABLE")
    print("="*60)
    print(summary_f1.round(4))
    
    summary_acc.to_csv(f'Results/af_ada_{attack_type}_accuracy_summary.csv')
    summary_f1.to_csv(f'Results/af_ada_{attack_type}_f1_summary.csv')
    
    return df_all


In [2]:
# ==================== MAIN ====================

if __name__ == "__main__":
    

    MODEL_SAVE_PATH = 'Models'
    
    # ========== مرحله 1: آموزش (فقط یک بار اجرا شود) ==========
    print("\n" + "🔷" * 40)
    print("PHASE 1: TRAINING")
    print("🔷" * 40)
    
    # آموزش مدل و ذخیره آن
    trained_model = train_model(
        save_model_path=MODEL_SAVE_PATH
    )
   


🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷
PHASE 1: TRAINING
🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷🔷

TRAINING BLOCK - AF Model with ADA (Adversarial Domain Adaptation)

[1] Loading training data...
  Train: 3000, Val: 1000

[2] Preprocessing data...

[3] Initializing AF Model...
  Input dimension: 18
  Number of classes: 5
  Embedding size: 512
  Hidden dims: [256, 128]
  Class loss weight: 4.0, Domain loss weight: 4.0
  Learning rates: Classifier=1e-4, Discriminator=1e-5, Combined=1e-5

[4] Training model...
  Model initialized: input_dim=18, num_classes=5
  Training: 3000 samples, batch_size=32, epochs=80
  Epoch  20: Class Loss=1.0026, Domain Comb=0.7089, Domain Disc=0.7173
  Epoch  40: Class Loss=0.9367, Domain Comb=0.7085, Domain Disc=0.7111
  Epoch  60: Class Loss=0.9104, Domain Comb=0.7098, Domain Disc=0.7113
  Epoch  80: Class Loss=0.8886, Domain Comb=0.7104, Domain Disc=0.7067

[5] Evaluating on validation set...

VALIDATION RESULTS
  Accuracy:  0.7260
  Precision: 0.7513
  R

In [8]:
# ==================== MAIN For Test ====================

def main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET):
    

    MODEL_SAVE_PATH = 'Models'
    

    # ========== مرحله 2: تست (می‌توان چند بار با تنظیمات مختلف اجرا کرد) ==========
    print("\n" + "🔶" * 40)
    print("PHASE 2: TESTING")
    print("🔶" * 40)
    
    # تست با یک بودجه مشخص
    results_single = test_model(
        attack_type=ATTACK_TYPE,
        test_perturb_levels=TEST_PERTURB_LEVELS,
        update_budget=UPDATE_BUDGET,
        model_path=MODEL_SAVE_PATH
    )
    
    print("\n" + "="*80)
    print("ALL OPERATIONS COMPLETED")
    print("="*80)

In [4]:
if __name__ == "__main__":

    # ========== تنظیمات ==========
    ATTACK_TYPE = 'FGSM'  # گزینه‌ها: 'DeepFool', 'PGD', 'CW'
    TEST_PERTURB_LEVELS = [0.01, 0.05, 0.1, 0.2, 0.5]
    UPDATE_BUDGET = 0.2  # 20% بودجه برای به‌روزرسانی
    main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET)


🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶
PHASE 2: TESTING
🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶

TESTING BLOCK - AF Model with ADA
Attack Type: FGSM, Update Budget: 20.0%

[1] Loading pre-trained model...
  Model initialized: input_dim=18, num_classes=5
  Model loaded from Models/

[3] Loading perturbed data (attack type: FGSM)...
  Loaded FGSM eps=0.01: 1000 samples
  Loaded FGSM eps=0.05: 1000 samples
  Loaded FGSM eps=0.1: 1000 samples
  Loaded FGSM eps=0.2: 1000 samples
  Loaded FGSM eps=0.5: 1000 samples

[4] Running online tests with 20.0% update budget...

--- Testing Perturbation Level 0.01 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent Acc=0.7280
  Final - Acc: 0.7280, Precision: 0.7537, Recall: 0.7280, F1: 0.7258

--- Testing Perturbation Level 0.05 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent Acc=0.7130
  Final - Acc: 0.7130, Precision: 0.7383, Recall: 0.7130, F1: 0.7105

--- Testing Perturbation Level 0.1 (Budget: 20.0%) ---
  Sample 1000: Updates=200,

In [5]:
if __name__ == "__main__":
        # ========== تنظیمات ==========
    ATTACK_TYPE = 'PGD'  # گزینه‌ها: 'DeepFool', 'PGD', 'CW'
    TEST_PERTURB_LEVELS = [0.01, 0.05, 0.1, 0.2, 0.5]
    UPDATE_BUDGET = 0.2  # 20% بودجه برای به‌روزرسانی
    main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET)


🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶
PHASE 2: TESTING
🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶

TESTING BLOCK - AF Model with ADA
Attack Type: PGD, Update Budget: 20.0%

[1] Loading pre-trained model...
  Model initialized: input_dim=18, num_classes=5
  Model loaded from Models/

[3] Loading perturbed data (attack type: PGD)...
  Loaded PGD eps=0.01: 1000 samples
  Loaded PGD eps=0.05: 1000 samples
  Loaded PGD eps=0.1: 1000 samples
  Loaded PGD eps=0.2: 1000 samples
  Loaded PGD eps=0.5: 1000 samples

[4] Running online tests with 20.0% update budget...

--- Testing Perturbation Level 0.01 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent Acc=0.7300
  Final - Acc: 0.7300, Precision: 0.7576, Recall: 0.7300, F1: 0.7284

--- Testing Perturbation Level 0.05 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent Acc=0.7080
  Final - Acc: 0.7080, Precision: 0.7355, Recall: 0.7080, F1: 0.7068

--- Testing Perturbation Level 0.1 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent

In [6]:
if __name__ == "__main__":
        # ========== تنظیمات ==========
    ATTACK_TYPE = 'CW'  # گزینه‌ها: 'DeepFool', 'PGD', 'CW'
    TEST_PERTURB_LEVELS = [0.1, 0.5, 1.0, 2.0, 5.0]
    UPDATE_BUDGET = 0.2  # 20% بودجه برای به‌روزرسانی
    main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET)


🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶
PHASE 2: TESTING
🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶

TESTING BLOCK - AF Model with ADA
Attack Type: CW, Update Budget: 20.0%

[1] Loading pre-trained model...
  Model initialized: input_dim=18, num_classes=5
  Model loaded from Models/

[3] Loading perturbed data (attack type: CW)...
  Loaded CW eps=0.1: 1000 samples
  Loaded CW eps=0.5: 1000 samples
  Loaded CW eps=1.0: 1000 samples
  Loaded CW eps=2.0: 1000 samples
  Loaded CW eps=5.0: 1000 samples

[4] Running online tests with 20.0% update budget...

--- Testing Perturbation Level 0.1 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent Acc=0.7280
  Final - Acc: 0.7280, Precision: 0.7551, Recall: 0.7280, F1: 0.7261

--- Testing Perturbation Level 0.5 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent Acc=0.6910
  Final - Acc: 0.6910, Precision: 0.7147, Recall: 0.6910, F1: 0.6873

--- Testing Perturbation Level 1.0 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent Acc=0.6270

In [7]:
if __name__ == "__main__":
        # ========== تنظیمات ==========
    ATTACK_TYPE = 'DeepFool'  # گزینه‌ها: 'DeepFool', 'PGD', 'CW'
    TEST_PERTURB_LEVELS = [0.1, 0.5, 1.0, 2.0, 5.0]
    UPDATE_BUDGET = 0.2  # 20% بودجه برای به‌روزرسانی
    main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET)


🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶
PHASE 2: TESTING
🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶🔶

TESTING BLOCK - AF Model with ADA
Attack Type: DeepFool, Update Budget: 20.0%

[1] Loading pre-trained model...
  Model initialized: input_dim=18, num_classes=5
  Model loaded from Models/

[3] Loading perturbed data (attack type: DeepFool)...
  Loaded DeepFool eps=0.1: 1000 samples
  Loaded DeepFool eps=0.5: 1000 samples
  Loaded DeepFool eps=1.0: 1000 samples
  Loaded DeepFool eps=2.0: 1000 samples
  Loaded DeepFool eps=5.0: 1000 samples

[4] Running online tests with 20.0% update budget...

--- Testing Perturbation Level 0.1 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent Acc=0.7320
  Final - Acc: 0.7320, Precision: 0.7583, Recall: 0.7320, F1: 0.7303

--- Testing Perturbation Level 0.5 (Budget: 20.0%) ---
  Sample 1000: Updates=200, Recent Acc=0.7140
  Final - Acc: 0.7140, Precision: 0.7319, Recall: 0.7140, F1: 0.7117

--- Testing Perturbation Level 1.0 (Budget: 20.0%) ---
  S